# Phase 3 – Churn KPI Calculations

## Objective

The objective of this phase is to calculate the key business KPIs used by management to monitor customer churn, revenue, customer satisfaction, and business performance.

These KPIs help evaluate customer retention, revenue growth, contract performance, and service quality.

In [1]:
import pandas as pd
import numpy as np

In [2]:
file_path = r"C:\Users\Mayuri\Desktop\NexaTel Cleanfiles\All_Excel_Converted_Into_Sheets.xlsx"
excel = pd.ExcelFile(file_path)

dfs = {}

for sheet in excel.sheet_names:
    dfs[sheet] = excel.parse(sheet)

print(excel.sheet_names)

['billing_cleaned', 'cities_cleaned', 'complaints_cleaned', 'contracts_cleaned', 'customers_cleaned', 'customer_feedback_cleaned', 'data_quality_issue_log_cleaned', 'devices_cleaned', 'employees_cleaned', 'marketing_campaigns_cleaned', 'network_quality_cleaned', 'payments_cleaned', 'plans_cleaned', 'plan_history_cleaned', 'recharges_cleaned', 'regions_cleaned', 'retention_campaigns_cleaned', 'states_cleaned', 'stores_cleaned', 'subscriptions_cleaned', 'support_tickets_cleaned', 'usage_data_cleaned', 'usage_sms_cleaned', 'usage_voice_cleaned']


## Data Preparation

The required date columns were converted into datetime format before calculating business KPIs.

In [3]:
customers = dfs["customers_cleaned"]
billing = dfs["billing_cleaned"]
complaints = dfs["complaints_cleaned"]
contracts = dfs["contracts_cleaned"]

In [5]:
customers["Acquisition_Date"] = pd.to_datetime(
    customers["Acquisition_Date"],
    dayfirst=True,
    errors="coerce"
)

customers["Churn_Date"] = pd.to_datetime(
    customers["Churn_Date"],
    dayfirst=True,
    errors="coerce"
)

billing["Billing_Date"] = pd.to_datetime(
    billing["Billing_Date"],
    dayfirst=True,
    errors="coerce"
)

complaints["Complaint_Date"] = pd.to_datetime(
    complaints["Complaint_Date"],
    dayfirst=True,
    errors="coerce"
)

complaints["Resolution_Date"] = pd.to_datetime(
    complaints["Resolution_Date"],
    dayfirst=True,
    errors="coerce"
)

## KPI 1 – Customer Churn Rate

Customer Churn Rate measures the percentage of customers who discontinued the service during the analysis period.

Formula:

**Customers Lost / Customers at Start × 100**

In [6]:
customers_start = len(customers)

customers_lost = (customers["Customer_Status"] == "Churned").sum()

churn_rate = (customers_lost / customers_start) * 100

print("Customers at Start :", customers_start)
print("Customers Lost :", customers_lost)
print("Customer Churn Rate : {:.2f}%".format(churn_rate))

Customers at Start : 19000
Customers Lost : 3506
Customer Churn Rate : 18.45%


### Observation

The dataset contains **19,000 customers**, of which **3,506 customers** have churned.

The calculated **Customer Churn Rate is 18.45%**, indicating that a significant proportion of customers have discontinued the service. This churn rate is considerably higher than the desired business target of less than 2.1% per month, suggesting the need for improved customer retention strategies.

## KPI 2 – Customer Retention Rate

Customer Retention Rate measures the percentage of customers who continue using the service after excluding newly acquired customers.

Formula:

**(Customers at End − New Customers) / Customers at Start × 100**

In [7]:
customers_end = (customers["Customer_Status"] == "Active").sum()

new_customers = customers[
    customers["Acquisition_Date"].dt.year == customers["Acquisition_Date"].dt.year.max()
].shape[0]

retention_rate = ((customers_end - new_customers) / customers_start) * 100

print("Customers at End:", customers_end)
print("New Customers:", new_customers)
print("Customer Retention Rate: {:.2f}%".format(retention_rate))

Customers at End: 15494
New Customers: 203
Customer Retention Rate: 80.48%


### Observation

The analysis shows that **15,494 customers remained active**, while **203 customers were newly acquired** during the latest acquisition period.

The calculated **Customer Retention Rate is 80.48%**, indicating that the majority of customers continue using NexaTel's services. Although the retention rate is relatively strong, it also suggests that there is room for improvement to further reduce customer churn and strengthen long-term customer loyalty.

## KPI 3 – Average Revenue Per User (ARPU)

Average Revenue Per User (ARPU) measures the average monthly revenue generated from each active customer.

**Formula:**

**ARPU = Total Recurring Revenue / Average Active Subscribers**

This KPI helps evaluate customer monetization and is widely used to measure the revenue generated per active subscriber.

In [8]:
total_revenue = billing["Total_Amount_Inr"].sum()

active_customers = (customers["Customer_Status"] == "Active").sum()

arpu = total_revenue / active_customers

print("Total Revenue (INR):", round(total_revenue, 2))
print("Active Customers:", active_customers)
print("ARPU (INR):", round(arpu, 2))

Total Revenue (INR): 171128477.15
Active Customers: 15494
ARPU (INR): 11044.82


### Observation

The total recurring revenue generated from the billing dataset is **₹171,128,477.15**, with **15,494 active customers**.

The calculated **Average Revenue Per User (ARPU) is ₹11,044.82**, indicating that each active customer contributes approximately **₹11,045** in recurring revenue on average. A higher ARPU reflects better customer monetization and stronger revenue generation for the business.

## KPI 4 – Customer Lifetime Value (CLV)

Customer Lifetime Value (CLV) estimates the total revenue expected from an average customer throughout their relationship with the company.

**Formula:**

**CLV = ARPU × Gross Margin % × (1 / Churn Rate)**

**Assumption:** Since the dataset does not include Gross Margin %, a value of **70%** has been assumed for calculation purposes.

In [9]:
gross_margin = 0.70

clv = arpu * gross_margin * (1 / (churn_rate / 100))

print("Gross Margin:", gross_margin * 100, "%")
print("Customer Lifetime Value (INR):", round(clv, 2))

Gross Margin: 70.0 %
Customer Lifetime Value (INR): 41898.5


### Observation

Using an assumed **70% gross margin**, the estimated **Customer Lifetime Value (CLV)** is **₹41,898.50**.

This indicates that, on average, a customer is expected to generate approximately **₹41,899** in value over their relationship with NexaTel. Improving customer retention and reducing churn can further increase the overall customer lifetime value and business profitability.

## KPI 5 – Revenue Lost to Churn

Revenue Lost to Churn estimates the recurring revenue lost due to customers who have discontinued the service.

**Formula:**

**Revenue Lost = Churned Customers × ARPU**

In [10]:
revenue_lost = customers_lost * arpu

print("Churned Customers:", customers_lost)
print("ARPU (INR):", round(arpu, 2))
print("Revenue Lost to Churn (INR):", round(revenue_lost, 2))

Churned Customers: 3506
ARPU (INR): 11044.82
Revenue Lost to Churn (INR): 38723147.08


### Observation

The analysis shows that **3,506 customers** have churned, resulting in an estimated **Revenue Lost to Churn of ₹38,723,147.08**.

This indicates a substantial financial impact on the business. Reducing customer churn through improved customer satisfaction, proactive support, and targeted retention strategies can help recover a significant portion of recurring revenue.

## KPI 6 – Monthly Recurring Revenue (MRR)

Monthly Recurring Revenue (MRR) represents the predictable monthly revenue generated from active customers.

**Formula:**

**MRR = Sum of recurring charges for active customers**

MRR is one of the most important KPIs for subscription-based businesses because it measures stable and recurring income.

In [11]:
active_customers = customers[
    customers["Customer_Status"] == "Active"
][["Customer_Id"]]

active_billing = billing.merge(
    active_customers,
    on="Customer_Id",
    how="inner"
)

mrr = active_billing["Total_Amount_Inr"].sum()

print("Monthly Recurring Revenue (INR):", round(mrr, 2))

Monthly Recurring Revenue (INR): 152445826.89


### Observation:
- The calculated Monthly Recurring Revenue (MRR) is **₹152.45 Million (₹15.24 Crore)**.
- This indicates the total predictable revenue generated from active customers/subscriptions during the month.
- A high MRR value reflects a strong recurring customer base and stable revenue generation.
- Monitoring MRR over time helps track business growth, customer retention, and revenue stability.

In [18]:
# Step 9: Revenue by Customer Segment

# Merge customer details with billing revenue
customer_revenue = billing.merge(
    customers[['Customer_Id', 'Customer_Segment']],
    on='Customer_Id',
    how='left'
)

# Calculate revenue by segment
segment_revenue = customer_revenue.groupby('Customer_Segment')['Total_Amount_Inr'].sum()

print(segment_revenue)

Customer_Segment
Enterprise    71908920.58
Premium       87394652.40
Value         11824904.17
Name: Total_Amount_Inr, dtype: float64


### Observation:
- The **Premium segment contributes the highest revenue** with ₹87.39 Million, making it the most valuable customer segment.
- The **Enterprise segment generates ₹71.91 Million**, showing strong revenue contribution from business customers.
- The **Value segment contributes ₹11.82 Million**, indicating lower revenue compared to Premium and Enterprise customers.
- Focusing on retention and personalized services for Premium and Enterprise customers can help maintain stable revenue growth.

In [19]:
# Step 10: Churn Rate by Customer Segment

# Count total customers and churned customers by segment

segment_churn = customers.groupby('Customer_Segment').agg(
    Total_Customers=('Customer_Id', 'count'),
    Churned_Customers=('Customer_Status', lambda x: (x == 'Churned').sum())
)

# Calculate churn rate
segment_churn['Churn_Rate_%'] = (
    segment_churn['Churned_Customers'] /
    segment_churn['Total_Customers']
) * 100

print(segment_churn)

                  Total_Customers  Churned_Customers  Churn_Rate_%
Customer_Segment                                                  
Enterprise                   1390                166     11.942446
Mass                         2932                782     26.671214
Premium                      8482               1283     15.126149
Value                        6196               1275     20.577792


### Observation:
- The **Mass customer segment has the highest churn rate (26.67%)**, indicating a greater risk of customer loss.
- The **Value segment shows the second-highest churn rate (20.58%)**, requiring focused retention efforts.
- The **Premium segment has a moderate churn rate (15.13%)**, while contributing the highest revenue, making customer retention important.
- The **Enterprise segment has the lowest churn rate (11.94%)**, showing stronger customer loyalty.
- Targeted retention strategies should focus on Mass and Value segments to reduce overall churn.

In [20]:
# Step 11: Customer Status Distribution

status_distribution = customers['Customer_Status'].value_counts()

print(status_distribution)

Customer_Status
Active     15494
Churned     3506
Name: count, dtype: int64


### Observation:
- The dataset contains **15,494 active customers** and **3,506 churned customers**.
- Active customers represent the majority of the customer base, indicating a strong retention level.
- However, the presence of 3,506 churned customers highlights a significant opportunity to improve customer retention.
- Understanding the reasons behind churn can help develop targeted strategies to reduce customer loss.

In [21]:
customers['Product_Line'].unique()

<StringArray>
[ 'Prepaid Mobile', 'Fiber Broadband',      'Enterprise',      'Smart Home',
 'Postpaid Mobile',              '5G',     'Ott Bundles',   'Iot Solutions']
Length: 8, dtype: str

In [22]:
# Step 12: Churn by Product Line

product_churn = customers.groupby('Product_Line').agg(
    Total_Customers=('Customer_Id','count'),
    Churned_Customers=('Customer_Status', lambda x: (x=='Churned').sum())
)

product_churn['Churn_Rate_%'] = (
    product_churn['Churned_Customers'] /
    product_churn['Total_Customers']
) * 100

print(product_churn)

                 Total_Customers  Churned_Customers  Churn_Rate_%
Product_Line                                                     
5G                          1549                320     20.658489
Enterprise                   811                 50      6.165228
Fiber Broadband             2193                346     15.777474
Iot Solutions                579                116     20.034542
Ott Bundles                  917                207     22.573610
Postpaid Mobile             3753                649     17.292832
Prepaid Mobile              8831               1733     19.624052
Smart Home                   367                 85     23.160763


### Observation:
- The **Smart Home product line has the highest churn rate (23.16%)**, indicating a need for improved customer experience and retention strategies.
- **OTT Bundles show the second-highest churn rate (22.57%)**, suggesting possible issues with service value, pricing, or customer satisfaction.
- **5G and IoT Solutions also have higher churn rates (above 20%)**, requiring further investigation.
- The **Enterprise product line has the lowest churn rate (6.17%)**, showing strong customer loyalty and satisfaction.
- Prepaid Mobile has the largest customer base and contributes the highest number of churned customers (1,733), making it a key area for churn reduction efforts.

In [23]:
# Step 13: Revenue Lost Due to Churn

churned_customers = customers[
    customers['Customer_Status']=='Churned'
]

churn_revenue = billing.merge(
    churned_customers[['Customer_Id','Customer_Segment']],
    on='Customer_Id',
    how='inner'
)

revenue_loss_segment = churn_revenue.groupby(
    'Customer_Segment'
)['Total_Amount_Inr'].sum()

print(revenue_loss_segment)

Customer_Segment
Enterprise     3850391.67
Premium       12291822.58
Value          2540436.01
Name: Total_Amount_Inr, dtype: float64


### Observation:
- The **Premium segment has the highest revenue loss due to churn (₹12.29 Million)**, indicating that losing premium customers has the greatest financial impact.
- The **Enterprise segment lost ₹3.85 Million in revenue**, showing the importance of maintaining strong relationships with business customers.
- The **Value segment contributed the lowest revenue loss (₹2.54 Million)** compared to other segments.
- Retention strategies should prioritize Premium customers because they generate higher revenue and their churn creates a significant business impact.
- Reducing churn among high-value customers can help protect recurring revenue and improve overall profitability.

In [24]:
# Step 14: CLV Analysis

clv_segment = customers.groupby(
    'Customer_Segment'
)['Annual_Income_Inr'].mean()

print(clv_segment)

Customer_Segment
Enterprise    879193.478959
Mass          858611.788365
Premium       871689.858295
Value         876272.277627
Name: Annual_Income_Inr, dtype: float64


### Observation:
- The **Enterprise segment has the highest average annual income value (₹879,193)**, indicating strong customer value potential.
- The **Value segment shows a comparable average income value (₹876,272)**, suggesting these customers also represent valuable opportunities.
- The **Premium segment has an average annual income of ₹871,690**, making it an important segment due to its high revenue contribution and lower churn risk.
- The **Mass segment has the lowest average income value (₹858,612)** among all segments, but it represents a large customer base.
- Customer value analysis helps identify segments where personalized services and retention strategies can maximize long-term revenue.

In [25]:
# Step 15: Final KPI Summary

kpi_summary = pd.DataFrame({
    "Metric": [
        "Total Customers",
        "Active Customers",
        "Churned Customers",
        "Churn Rate (%)",
        "Total Revenue (INR)",
        "MRR (INR)",
        "ARPU (INR)",
        "Customer Lifetime Value (INR)"
    ],
    "Value": [
        customers_start,
        customers_end,
        customers_lost,
        round(churn_rate,2),
        round(total_revenue,2),
        round(mrr,2),
        round(arpu,2),
        round(clv,2)
    ]
})

kpi_summary

,Metric,Value
0,Total Customers,1.900000e+04
1,Active Customers,1.549400e+04
2,Churned Customers,3.506000e+03
3,Churn Rate (%),1.845000e+01
4,Total Revenue (INR),1.711285e+08
5,MRR (INR),1.524458e+08
6,ARPU (INR),1.104482e+04
7,Customer Lifetime Value (INR),4.189850e+04


### Observation:
- The NexaTel dataset contains **19,000 total customers**, out of which **15,494 customers are active** and **3,506 customers have churned**.
- The overall churn rate is **18.45%**, indicating that customer retention improvement is an important business priority.
- The company generated total revenue of **₹171.13 Million (₹17.11 Crore)** with a Monthly Recurring Revenue (MRR) of **₹152.45 Million (₹15.24 Crore)**.
- The average revenue generated per customer (ARPU) is **₹11,044.82**, showing the average customer contribution to revenue.
- The Customer Lifetime Value (CLV) is **₹41,898.50**, representing the estimated long-term value generated from each customer.
- These KPIs provide a complete overview of business performance, customer health, and revenue stability for the Power BI churn dashboard.